In [1]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
from cfb_upsets.features import add_features

df = pd.read_csv("../data/processed/games_clean_all.csv")
df = add_features(df)
df_model = df.dropna(subset=["upset"]).copy()

In [2]:
feature_cols = ["spread_magnitude", "is_home_underdog", "elo_diff", "rolling_win_pct_diff", "week"]

# One-hot encode conference separately, since it's categorical
conf_dummies = pd.get_dummies(df_model["underdog_conference"], prefix="conf")

X = pd.concat([df_model[feature_cols], conf_dummies], axis=1)
y = df_model["upset"]

train_mask = df_model["season"] <= 2024
test_mask = df_model["season"] == 2025

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train: {len(X_train)} games (2021-2024)")
print(f"Test:  {len(X_test)} games (2025)")

Train: 3134 games (2021-2024)
Test:  808 games (2025)


In [3]:
baseline_predictions = np.zeros(len(y_test))  # 0 = favorite wins, our "always predict favorite" baseline
baseline_accuracy = (baseline_predictions == y_test).mean()

print(f"Baseline accuracy (always predict favorite wins): {baseline_accuracy:.1%}")

Baseline accuracy (always predict favorite wins): 73.5%


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

logreg_preds = logreg.predict(X_test)
logreg_accuracy = accuracy_score(y_test, logreg_preds)

print(f"Logistic Regression accuracy: {logreg_accuracy:.1%}")

Logistic Regression accuracy: 73.3%


In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_preds)

print(f"Random Forest accuracy: {rf_accuracy:.1%}")

Random Forest accuracy: 73.4%


In [6]:
from sklearn.metrics import confusion_matrix, classification_report

print("Logistic Regression:")
print(confusion_matrix(y_test, logreg_preds))
print(classification_report(y_test, logreg_preds))

print("\nRandom Forest:")
print(confusion_matrix(y_test, rf_preds))
print(classification_report(y_test, rf_preds))

Logistic Regression:
[[583  11]
 [205   9]]
              precision    recall  f1-score   support

         0.0       0.74      0.98      0.84       594
         1.0       0.45      0.04      0.08       214

    accuracy                           0.73       808
   macro avg       0.59      0.51      0.46       808
weighted avg       0.66      0.73      0.64       808


Random Forest:
[[593   1]
 [214   0]]
              precision    recall  f1-score   support

         0.0       0.73      1.00      0.85       594
         1.0       0.00      0.00      0.00       214

    accuracy                           0.73       808
   macro avg       0.37      0.50      0.42       808
weighted avg       0.54      0.73      0.62       808



In [7]:
logreg_probs = logreg.predict_proba(X_test)[:, 1]
rf_probs = rf.predict_proba(X_test)[:, 1]

print("Logistic regression predicted probabilities - summary:")
print(pd.Series(logreg_probs).describe())

Logistic regression predicted probabilities - summary:
count    808.000000
mean       0.285531
std        0.147553
min        0.003505
25%        0.161328
50%        0.311021
75%        0.415974
max        0.572658
dtype: float64


In [8]:
calibration_df = pd.DataFrame({
    "predicted_prob": logreg_probs,
    "actual_upset": y_test.values
})

calibration_df["prob_bucket"] = pd.qcut(calibration_df["predicted_prob"], q=5)

calibration_check = calibration_df.groupby("prob_bucket")["actual_upset"].agg(["mean", "count"])
print(calibration_check)

                      mean  count
prob_bucket                      
(0.00251, 0.129]  0.055556    162
(0.129, 0.255]    0.186335    161
(0.255, 0.351]    0.296296    162
(0.351, 0.433]    0.354037    161
(0.433, 0.573]    0.432099    162
